# T17 — Function Calling Deep Dive

## Objective

Build at least three custom tools using OpenAI Function Calling.

### Custom Tools

1. Calculator Tool — performs mathematical calculations.
2. Knowledge Search Tool — searches information from a local knowledge base.
3. Employee Database Tool — retrieves employee information from a database.

### Workflow

User Question → LLM → Tool Selection → Function Execution → Tool Result → Final AI Response

# 📑 Notebook Structure

This notebook is organized into the following sections:

1. **Title and Objective**
   - Define the objective and workflow of the task.

2. **Install Required Libraries**
   - Install OpenAI and environment-management libraries.

3. **Import Libraries and Initialize OpenAI Client**
   - Import the required Python libraries.
   - Load the OpenAI API key securely.

4. **Custom Tool 1 — Calculator**
   - Build a tool to perform mathematical calculations.
   - Test the calculator independently.

5. **Custom Tool 2 — Knowledge Search**
   - Build a tool to retrieve information from a local knowledge base.
   - Test the knowledge-search function independently.

6. **Custom Tool 3 — Employee Database Search**
   - Create a sample employee database using SQLite.
   - Retrieve employee information based on location or department.
   - Test the database-search function independently.

7. **OpenAI Function-Calling Tool Schemas**
   - Describe all three tools to the language model using JSON schemas.

8. **Tool Execution Logic**
   - Connect the model-selected tool to the appropriate Python function.

9. **Function-Calling AI Assistant**
   - Allow the LLM to automatically select and execute the appropriate tool.

10. **Final Testing and Results**
    - Test the Calculator Tool.
    - Test the Knowledge Search Tool.
    - Test the Employee Database Tool.

11. **Conclusion**
    - Summarize the implementation and results.

### Step 2: Install the required libraries

In [2]:
!pip install -q openai python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 3: Import libraries and load the API key

In [30]:
import os
import json
import sqlite3
import ast
import operator

from dotenv import load_dotenv
from openai import OpenAI

In [26]:
import os

from dotenv import load_dotenv
from openai import OpenAI

# Force reload the updated key from .env
load_dotenv(
    override=True
)

api_key = os.getenv(
    "OPENAI_API_KEY"
)

print(
    "Key found:",
    api_key is not None
)

print(
    "Key prefix:",
    api_key[:8] + "..."
    if api_key
    else "No key found"
)

client = OpenAI(
    api_key=api_key
)

print(
    "OpenAI client initialized successfully!"
)

Key found: True
Key prefix: sk-proj-...
OpenAI client initialized successfully!



## Custom Tool 1 — Calculator

This tool safely evaluates mathematical expressions such as addition, subtraction, multiplication, division and powers.

In [4]:
allowed_operators = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg
}


def evaluate_expression(node):

    if isinstance(node, ast.Constant):
        return node.value

    if isinstance(node, ast.BinOp):

        left_value = evaluate_expression(node.left)
        right_value = evaluate_expression(node.right)

        operation = allowed_operators.get(type(node.op))

        if operation is None:
            raise ValueError("This mathematical operation is not supported.")

        return operation(left_value, right_value)

    if isinstance(node, ast.UnaryOp):

        operation = allowed_operators.get(type(node.op))

        if operation is None:
            raise ValueError("This unary operation is not supported.")

        return operation(evaluate_expression(node.operand))

    raise ValueError("Invalid mathematical expression.")


def calculator(expression):

    try:

        parsed_expression = ast.parse(
            expression,
            mode="eval"
        )

        result = evaluate_expression(
            parsed_expression.body
        )

        return {
            "expression": expression,
            "result": result
        }

    except Exception as error:

        return {
            "error": str(error)
        }

### Test Calculator Tool

In [5]:
calculator("(25 * 4) + 50")

{'expression': '(25 * 4) + 50', 'result': 150}

## 4. Custom Tool 2 — Knowledge Search

This tool searches a local knowledge base and retrieves relevant information about AI concepts.

In [ ]:
knowledge_base = [

    {
        "title": "Retrieval-Augmented Generation",
        "content": (
            "RAG combines information retrieval with a large language "
            "model. Relevant documents are retrieved and provided to "
            "the model as context."
        )
    },

    {
        "title": "LangChain",
        "content": (
            "LangChain is a framework used to build applications "
            "powered by large language models. It supports prompts, "
            "retrievers, tools, agents and chains."
        )
    },

    {
        "title": "Embeddings",
        "content": (
            "Embeddings are numerical vector representations of text. "
            "Semantically similar text usually has similar vectors."
        )
    },

    {
        "title": "Function Calling",
        "content": (
            "Function calling allows a language model to select and "
            "request external tools using structured arguments."
        )
    },

    {
        "title": "ReAct Agent",
        "content": (
            "A ReAct agent combines reasoning and actions. The agent "
            "selects tools, observes their outputs and continues until "
            "it can answer the user."
        )
    }
]


def search_knowledge_base(query):

    query_words = set(
        query.lower().split()
    )

    results = []

    for document in knowledge_base:

        searchable_text = (
            document["title"]
            + " "
            + document["content"]
        ).lower()

        score = sum(
            word in searchable_text
            for word in query_words
        )

        if score > 0:

            results.append({
                "title": document["title"],
                "content": document["content"],
                "score": score
            })

    results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return results[:3]


### Test Knowledge Search Tool

In [7]:
search_knowledge_base(
    "What is function calling?"
)

[{'title': 'LangChain',
  'content': 'LangChain is a framework used to build applications powered by large language models. It supports prompts, retrievers, tools, agents and chains.',
  'score': 1},
 {'title': 'Function Calling',
  'content': 'Function calling allows a language model to select and request external tools using structured arguments.',
  'score': 1}]

## 5. Custom Tool 3 — Employee Database Search

This tool uses SQLite to store employee information and retrieve employees based on department or location.

In [12]:
import sqlite3

# Create connection to database
connection = sqlite3.connect("employee_database.db")

# Create cursor
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    role TEXT,
    location TEXT
)
""")

employee_data = [
    (101, "Aarav Sharma", "Data Science", "Data Scientist", "Pune"),
    (102, "Neha Patil", "Engineering", "Python Developer", "Mumbai"),
    (103, "Rohan Deshmukh", "AI", "Machine Learning Engineer", "Pune"),
    (104, "Sneha Joshi", "Human Resources", "HR Executive", "Nashik"),
    (105, "Arjun Mehta", "Engineering", "Backend Developer", "Bengaluru")
]

cursor.executemany(
    """
    INSERT OR REPLACE INTO employees
    VALUES (?, ?, ?, ?, ?)
    """,
    employee_data
)

connection.commit()

print("Employee database created successfully!")

Employee database created successfully!


### Employee Search Function

In [13]:
def search_employees(
    department=None,
    location=None
):

    query = """
    SELECT
        employee_id,
        name,
        department,
        role,
        location

    FROM employees

    WHERE 1 = 1
    """

    parameters = []

    if department:

        query += """
        AND LOWER(department)
        = LOWER(?)
        """

        parameters.append(
            department
        )

    if location:

        query += """
        AND LOWER(location)
        = LOWER(?)
        """

        parameters.append(
            location
        )

    cursor.execute(
        query,
        parameters
    )

    rows = cursor.fetchall()

    employees = []

    for row in rows:

        employees.append({

            "employee_id": row[0],

            "name": row[1],

            "department": row[2],

            "role": row[3],

            "location": row[4]
        })

    return employees

### Test Employee Database Tool

In [14]:
search_employees(
    location="Pune"
)

[{'employee_id': 101,
  'name': 'Aarav Sharma',
  'department': 'Data Science',
  'role': 'Data Scientist',
  'location': 'Pune'},
 {'employee_id': 103,
  'name': 'Rohan Deshmukh',
  'department': 'AI',
  'role': 'Machine Learning Engineer',
  'location': 'Pune'}]

## 7. OpenAI Function-Calling Tool Schemas

The following schemas describe the available custom tools to the language model.

Each schema defines:

- The name of the tool
- The purpose of the tool
- The input parameters accepted by the tool

The language model uses these descriptions to automatically select the appropriate tool and generate structured arguments.

In [15]:
tools = [

    # Tool 1: Calculator
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Performs mathematical calculations. "
                "Use this tool when the user asks to calculate "
                "or evaluate a mathematical expression."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": (
                            "The mathematical expression to evaluate. "
                            "Example: (25 * 4) + 50"
                        )
                    }
                },
                "required": [
                    "expression"
                ],
                "additionalProperties": False
            }
        }
    },

    # Tool 2: Knowledge Search
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": (
                "Searches the local knowledge base for information "
                "about artificial intelligence, RAG, LangChain, "
                "embeddings, function calling and ReAct agents."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": (
                            "The topic or question to search for "
                            "in the local knowledge base."
                        )
                    }
                },
                "required": [
                    "query"
                ],
                "additionalProperties": False
            }
        }
    },

    # Tool 3: Employee Database Search
    {
        "type": "function",
        "function": {
            "name": "search_employees",
            "description": (
                "Searches the employee database using a department, "
                "location or both. Use this tool when the user asks "
                "for employee information."
            ),
            "parameters": {
                "type": "object",
                "properties": {

                    "department": {
                        "type": "string",
                        "description": (
                            "The employee department, such as "
                            "Engineering, Data Science or AI."
                        )
                    },

                    "location": {
                        "type": "string",
                        "description": (
                            "The employee location, such as "
                            "Pune, Mumbai, Nashik or Bengaluru."
                        )
                    }
                },
                "additionalProperties": False
            }
        }
    }
]

print(
    f"{len(tools)} tool schemas created successfully!"
)

3 tool schemas created successfully!


In [16]:
for tool in tools:

    print(
        "Tool:",
        tool["function"]["name"]
    )

Tool: calculator
Tool: search_knowledge_base
Tool: search_employees


## 8. Tool Execution Logic

This section connects the tool selected by the language model to the corresponding Python function.

The function receives:

- The tool name selected by the LLM
- The structured arguments generated by the LLM

It then executes the appropriate custom Python function and returns the result.

In [17]:
def execute_tool(tool_name, tool_arguments):

    if tool_name == "calculator":

        return calculator(
            expression=tool_arguments["expression"]
        )

    elif tool_name == "search_knowledge_base":

        return search_knowledge_base(
            query=tool_arguments["query"]
        )

    elif tool_name == "search_employees":

        return search_employees(
            department=tool_arguments.get("department"),
            location=tool_arguments.get("location")
        )

    else:

        return {
            "error": f"Unknown tool: {tool_name}"
        }


print("Tool execution logic created successfully!")

Tool execution logic created successfully!


## Test Tool Execution Logic

The following tests verify that the execution function correctly calls each custom Python tool.

In [18]:
# Test Calculator Tool

calculator_result = execute_tool(
    "calculator",
    {
        "expression": "250 * 18"
    }
)

print(
    "Calculator Result:",
    calculator_result
)

Calculator Result: {'expression': '250 * 18', 'result': 4500}


In [19]:
# Test Knowledge Search Tool

knowledge_result = execute_tool(
    "search_knowledge_base",
    {
        "query": "What is RAG?"
    }
)

print(
    "Knowledge Search Result:",
    knowledge_result
)

Knowledge Search Result: [{'title': 'LangChain', 'content': 'LangChain is a framework used to build applications powered by large language models. It supports prompts, retrievers, tools, agents and chains.', 'score': 1}]


In [20]:
# Test Employee Database Tool

employee_result = execute_tool(
    "search_employees",
    {
        "location": "Pune"
    }
)

print(
    "Employee Search Result:",
    employee_result
)

Employee Search Result: [{'employee_id': 101, 'name': 'Aarav Sharma', 'department': 'Data Science', 'role': 'Data Scientist', 'location': 'Pune'}, {'employee_id': 103, 'name': 'Rohan Deshmukh', 'department': 'AI', 'role': 'Machine Learning Engineer', 'location': 'Pune'}]


## 9. Function-Calling AI Assistant

This section integrates the custom tools with the OpenAI language model.

The assistant performs the following steps:

1. Receives a question from the user.
2. Analyzes the available tool schemas.
3. Automatically selects the appropriate tool.
4. Generates structured arguments for the selected tool.
5. Executes the corresponding Python function.
6. Sends the tool result back to the language model.
7. Generates a final natural-language response.

In [23]:
def function_calling_assistant(user_question):

    # Step 1: Send the user's question
    # and available tools to the LLM

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant. "
                "Use the available tools whenever required. "
                "Select the most appropriate tool based "
                "on the user's question."
            )
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    first_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    assistant_message = (
        first_response
        .choices[0]
        .message
    )

    # Step 2: Check whether the LLM
    # requested a tool

    if assistant_message.tool_calls:

        messages.append(
            assistant_message
        )

        # Step 3: Execute every requested tool

        for tool_call in assistant_message.tool_calls:

            tool_name = (
                tool_call
                .function
                .name
            )

            tool_arguments = json.loads(
                tool_call
                .function
                .arguments
            )

            print(
                "Selected Tool:",
                tool_name
            )

            print(
                "Tool Arguments:",
                tool_arguments
            )

            # Execute the selected Python tool

            tool_result = execute_tool(
                tool_name,
                tool_arguments
            )

            print(
                "Tool Result:",
                tool_result
            )

            # Step 4: Send the tool result
            # back to the LLM

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(
                        tool_result
                    )
                }
            )

        # Step 5: Generate the final answer

        final_response = (
            client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages
            )
        )

        return (
            final_response
            .choices[0]
            .message
            .content
        )

    # If no tool is required,
    # return the model's normal response

    return assistant_message.content


print(
    "Function-calling assistant created successfully!"
)

Function-calling assistant created successfully!


### Test 1 — Calculator Tool

This test checks whether the language model automatically selects and executes the Calculator Tool.

In [27]:
response = function_calling_assistant(
    "Calculate 250 multiplied by 18."
)

print(
    "\nFinal AI Response:"
)

print(
    response
)

Selected Tool: calculator
Tool Arguments: {'expression': '250 * 18'}
Tool Result: {'expression': '250 * 18', 'result': 4500}

Final AI Response:
250 multiplied by 18 is 4500.


### Test 2 — Knowledge Search Tool

This test checks whether the language model automatically selects the Knowledge Search Tool.

In [28]:
response = function_calling_assistant(
    "What is Retrieval-Augmented Generation?"
)

print(
    "\nFinal AI Response:"
)

print(
    response
)

Selected Tool: search_knowledge_base
Tool Arguments: {'query': 'Retrieval-Augmented Generation'}
Tool Result: [{'title': 'Retrieval-Augmented Generation', 'content': 'RAG combines information retrieval with a large language model. Relevant documents are retrieved and provided to the model as context.', 'score': 2}]

Final AI Response:
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with large language models. In this approach, relevant documents are retrieved from a database and provided to the language model as context, enhancing the model's ability to generate accurate and contextually relevant responses. This method leverages the strengths of both retrieval systems and generative models to improve performance in tasks that require up-to-date or specific information.


### Test 3 — Employee Database Tool

This test checks whether the language model automatically selects the Employee Database Search Tool.

In [29]:
response = function_calling_assistant(
    "Find all employees located in Pune."
)

print(
    "\nFinal AI Response:"
)

print(
    response
)

Selected Tool: search_employees
Tool Arguments: {'location': 'Pune'}
Tool Result: [{'employee_id': 101, 'name': 'Aarav Sharma', 'department': 'Data Science', 'role': 'Data Scientist', 'location': 'Pune'}, {'employee_id': 103, 'name': 'Rohan Deshmukh', 'department': 'AI', 'role': 'Machine Learning Engineer', 'location': 'Pune'}]

Final AI Response:
Here are the employees located in Pune:

1. **Aarav Sharma**
   - Employee ID: 101
   - Department: Data Science
   - Role: Data Scientist

2. **Rohan Deshmukh**
   - Employee ID: 103
   - Department: AI
   - Role: Machine Learning Engineer


## 10. Conclusion

In this task, three custom tools were developed and integrated using OpenAI Function Calling:

1. **Calculator Tool** — performs mathematical calculations.
2. **Knowledge Search Tool** — retrieves relevant information from a local knowledge base.
3. **Employee Database Tool** — searches employee information using location or department.

The OpenAI language model automatically analyzes the user's question, selects the appropriate tool, generates structured arguments, executes the corresponding Python function and uses the tool result to generate a final natural-language response.

This implementation demonstrates how function calling allows a large language model to interact with external tools and perform tasks beyond normal text generation.